In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold, cross_val_score
import plotly.graph_objects as go

In [ ]:
def build_nextx_dataset(
    df_long: pd.DataFrame,
    sid_col="sid", landmark_col="landmark",
    age_col="age", x_col="x",
    use_dt=True
):
    """
    Returns:
        X (ndarray): features [x_t,(dt)] per row
        y (ndarray): targets x_{t+1}
        groups (ndarray): group ids for GroupKFold (by sid)
        meta (DataFrame): rows with sid, landmark, age_t, age_tp1, x_t, x_tp1
    """
    cols_needed = {sid_col, landmark_col, age_col, x_col}
    missing = cols_needed - set(df_long.columns)
    if missing:
        raise ValueError(f"Missing columns: {missing}")
    d = df_long[[sid_col, landmark_col, age_col, x_col]].dropna().copy()
    d[age_col] = pd.to_numeric(d[age_col], errors="coerce")
    d[x_col] = pd.to_numeric(d[x_col], errors="coerce")
    d = d.dropna(subset=[age_col, x_col])

    rows = []
    for (sid, lmk), g in d.groupby([sid_col, landmark_col], sort=False):
        g = g.sort_values(age_col)
        xs = g[x_col].to_numpy()
        ages = g[age_col].to_numpy()
        if len(xs) < 2:
            continue
        x_t   = xs[:-1]
        x_tp1 = xs[1:]
        dt    = np.diff(ages)
        # Keep only positive/nonzero steps
        mask = np.isfinite(x_t) & np.isfinite(x_tp1) & np.isfinite(dt) & (dt > 0)
        if not np.any(mask):
            continue
        for xt, xt1, dti, a_t, a_tp1 in zip(x_t[mask], x_tp1[mask], dt[mask], ages[:-1][mask], ages[1:][mask]):
            feats = [xt] + ([dti] if use_dt else [])
            rows.append((sid, lmk, a_t, a_tp1, xt, xt1, *feats))

    if not rows:
        raise ValueError("No usable (x_t → x_{t+1}) pairs found. Check your data.")

    # Assemble
    cols = [sid_col, landmark_col, f"{age_col}_t", f"{age_col}_tp1", "x_t", "x_tp1", "feat_x_t"] + (["feat_dt"] if use_dt else [])
    meta = pd.DataFrame(rows, columns=cols)
    y = meta["x_tp1"].to_numpy().astype(float)
    if use_dt:
        X = meta[["feat_x_t", "feat_dt"]].to_numpy(dtype=float)
    else:
        X = meta[["feat_x_t"]].to_numpy(dtype=float)
    groups = meta[sid_col].astype(str).to_numpy()
    return X, y, groups, meta

# ----------------------------
# 2) Train a simple model (Ridge). Swap for any regressor.
# ----------------------------
def fit_nextx_regressor(df_long, use_dt=True, alpha=1.0, cv_splits=5,x_col='x'):
    X, y, groups, meta = build_nextx_dataset(df_long,x_col=x_col, use_dt=use_dt)
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=alpha, random_state=0))
    ])
    # Subject-wise CV
    gkf = GroupKFold(n_splits=min(cv_splits, len(np.unique(groups))))
    scores = cross_val_score(pipe, X, y, groups=groups, cv=gkf, scoring="neg_mean_absolute_error")
    pipe.fit(X, y)
    return pipe, {"cv_MAE(mean)": -scores.mean(), "cv_MAE(std)": scores.std(), "n_pairs": len(y)}, meta

# ----------------------------
# 3) Predict the next x from a current x (and dt)
# ----------------------------
def predict_next_x(model, x_t, dt=None):
    if dt is None:
        X = np.array([[x_t]], float)
    else:
        X = np.array([[x_t, dt]], float)
    return float(model.predict(X)[0])

In [51]:
import numpy as np
import plotly.graph_objects as go

def plot_subject_landmark_nextx(df_long, model, sid, landmark, use_dt=True,
                                sid_col="sid", landmark_col="landmark", age_col="age", x_col="x",
                                circle_mm=2.0):
    g = df_long.loc[(df_long[sid_col] == sid) & (df_long[landmark_col] == landmark),
                    [age_col, x_col]].dropna().copy()
    if g.empty or g.shape[0] < 2:
        raise ValueError("Need at least 2 points for this subject/landmark.")
    g[age_col] = pd.to_numeric(g[age_col], errors="coerce")
    g = g.dropna().sort_values(age_col)
    ages = g[age_col].to_numpy(float)
    xs   = g[x_col].to_numpy(float)

    # One-step-ahead predictions using observed x_t (not rolling)
    dt = np.diff(ages) if use_dt else None
    preds = []
    for i in range(len(xs) - 1):
        xt = xs[i]
        dti = dt[i] if use_dt else None
        preds.append(predict_next_x(model, xt, dti))
    ages_tp1 = ages[1:]

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=ages, y=xs, mode="lines+markers", name="Actual x(age)"))
    fig.add_trace(go.Scatter(x=ages_tp1, y=preds, mode="markers", name="Predicted x_{t+1}"))


    tick_mm = 4.0  # total length in mm; change to what you want

    for ax, py in zip(ages_tp1, preds):
        fig.add_shape(
            type="line",
            x0=ax, x1=ax,                     # at age_{t+1}
            y0=py - tick_mm/2, y1=py + tick_mm/2,  # small vertical segment
            line=dict(width=2, dash="dot"),
            xref="x", yref="y",
            layer="above",
        )


    fig.update_layout(
        title=f"Next-step prediction (sid={sid}, landmark={landmark})",
        xaxis_title="Age",
        yaxis_title="x",
        template="plotly_white",
        height=450
    )
    return fig


In [4]:
df_long =pd.read_csv("/data/all_landmark_series_long.csv") 
male_df = df_long[df_long['sex']=='M']
female_df = df_long[df_long['sex']=='F']

In [6]:
# ----------------------------
# Example usage
# ----------------------------
 # needs columns: sid, landmark, age, x
model, info, meta = fit_nextx_regressor(male_df, use_dt=True, alpha=1.0, cv_splits=5)
print(info)  # {'cv_MAE(mean)': ..., 'cv_MAE(std)': ..., 'n_pairs': ...}
model_f, info_f, meta_f = fit_nextx_regressor(female_df, use_dt=True, alpha=1.0, cv_splits=5)
print(info_f)


{'cv_MAE(mean)': 3.0733469402188596, 'cv_MAE(std)': 0.3415098033228238, 'n_pairs': 18899}
{'cv_MAE(mean)': 1.3839601328015683, 'cv_MAE(std)': 0.043992110891793966, 'n_pairs': 21160}


In [52]:
# Quick check plot for one series:
fig = plot_subject_landmark_nextx(df_long, model, sid="U506", landmark="ans", use_dt=True)
fig.show()
# Predict a single next step:
x_next_hat = predict_next_x(model, x_t=42.3, dt=1.0)
print("x_next_hat =", x_next_hat)

x_next_hat = 49.02646101438905


In [53]:
landmarks = ['nasion', 'point a','point b','mid gonion','ans','articular','pogonion']
for l in landmarks:
    print(l)
    ans_male = male_df[male_df['landmark']==l]
    model, info, meta = fit_nextx_regressor(ans_male, use_dt=True, alpha=1.0, cv_splits=5)
    print(info) 
    try:
        fig = plot_subject_landmark_nextx(ans_male, model, sid="U506", landmark=l, use_dt=True)
        fig.show()
    except:
        print("landmark: ", l, "sid: ","U506")


nasion
{'cv_MAE(mean)': 0.787017187769543, 'cv_MAE(std)': 0.08409872377619344, 'n_pairs': 781}


point a
{'cv_MAE(mean)': 1.0428717702392023, 'cv_MAE(std)': 0.06803715771580732, 'n_pairs': 775}


point b
{'cv_MAE(mean)': 1.5430991216144743, 'cv_MAE(std)': 0.13544327522776434, 'n_pairs': 773}


mid gonion
{'cv_MAE(mean)': 1.598977105653828, 'cv_MAE(std)': 0.10584742945223838, 'n_pairs': 314}
landmark:  mid gonion sid:  U506
ans
{'cv_MAE(mean)': 1.401813450332243, 'cv_MAE(std)': 0.11063587632209666, 'n_pairs': 776}


articular
{'cv_MAE(mean)': 1.0075566358445696, 'cv_MAE(std)': 0.09821073879285867, 'n_pairs': 763}


pogonion
{'cv_MAE(mean)': 5.907857813542762, 'cv_MAE(std)': 1.8850778438186506, 'n_pairs': 775}


In [54]:
landmarks = ['nasion', 'point a','point b','mid gonion','ans','articular','pogonion']
for l in landmarks:
    print(l)
    ans_female = female_df[female_df['landmark']==l]
    model, info, meta = fit_nextx_regressor(ans_female, use_dt=True, alpha=1.0, cv_splits=5)
    print(info) 
    fig = plot_subject_landmark_nextx(ans_female, model, sid="005", landmark=l, use_dt=True)
    fig.show()

nasion
{'cv_MAE(mean)': 0.7174038348327576, 'cv_MAE(std)': 0.05770992123308558, 'n_pairs': 877}


point a
{'cv_MAE(mean)': 1.0254704042363838, 'cv_MAE(std)': 0.07190528092643401, 'n_pairs': 871}


point b
{'cv_MAE(mean)': 1.566160923128152, 'cv_MAE(std)': 0.09562876110610856, 'n_pairs': 872}


mid gonion
{'cv_MAE(mean)': 1.605840739416029, 'cv_MAE(std)': 0.11584155865687394, 'n_pairs': 358}


ans
{'cv_MAE(mean)': 1.3147086423334073, 'cv_MAE(std)': 0.16063773742838827, 'n_pairs': 874}


articular
{'cv_MAE(mean)': 0.847702960813838, 'cv_MAE(std)': 0.0984226092598928, 'n_pairs': 873}


pogonion
{'cv_MAE(mean)': 1.718370182620029, 'cv_MAE(std)': 0.15638008061304065, 'n_pairs': 874}


In [56]:
def plot_subject_landmark_nextxy(df_long, model_x, model_y, sid, landmark, use_dt=True,
                                 sid_col="sid", landmark_col="landmark",
                                 age_col="age", x_col="x", y_col="y",
                                 circle_mm=2.0):
    # --- data prep ---
    g = df_long.loc[
        (df_long[sid_col] == sid) & (df_long[landmark_col] == landmark),
        [age_col, x_col, y_col]
    ].dropna().copy()
    if g.empty or g.shape[0] < 2:
        raise ValueError("Need at least 2 rows for this subject/landmark.")

    g[age_col] = pd.to_numeric(g[age_col], errors="coerce")
    g[x_col]   = pd.to_numeric(g[x_col], errors="coerce")
    g[y_col]   = pd.to_numeric(g[y_col], errors="coerce")
    g = g.dropna().sort_values(age_col)

    ages = g[age_col].to_numpy(float)
    xs   = g[x_col].to_numpy(float)
    ys   = g[y_col].to_numpy(float)

    dt = np.diff(ages) if use_dt else None

    # --- one-step-ahead predictions (x_{t+1}, y_{t+1}) ---
    px, py = [], []
    for i in range(len(xs) - 1):
        dti = float(dt[i]) if use_dt else None
        px.append(predict_next_x(model_x, xs[i], dti))
        py.append(predict_next_x(model_y, ys[i], dti))
    px = np.asarray(px, float)
    py = np.asarray(py, float)
    ages_tp1 = ages[1:]

    # --- figure ---
    fig = go.Figure()

    # actual XY path over time
    fig.add_trace(go.Scatter(
        x=xs, y=ys, mode="lines+markers",
        name="Actual (x,y) path",
        text=[f"age={a}" for a in ages],
        hovertemplate="x=%{x:.2f} mm<br>y=%{y:.2f} mm<br>%{text}<extra></extra>"
    ))

    # predicted points at t+1
    fig.add_trace(go.Scatter(
        x=px, y=py, mode="markers",
        name="Predicted (x,y) at next age",
        text=[f"age={a}" for a in ages_tp1],
        marker=dict(symbol="x", size=9),
        hovertemplate="<b>pred</b> x=%{x:.2f}, y=%{y:.2f} mm<br>%{text}<extra></extra>"
    ))

    # axis ranges with padding to fit circles
    xmin = float(np.nanmin([xs.min(), px.min() if len(px) else xs.min()])) - circle_mm*1.2
    xmax = float(np.nanmax([xs.max(), px.max() if len(px) else xs.max()])) + circle_mm*1.2
    ymin = float(np.nanmin([ys.min(), py.min() if len(py) else ys.min()])) - circle_mm*1.2
    ymax = float(np.nanmax([ys.max(), py.max() if len(py) else ys.max()])) + circle_mm*1.2

    # 2 mm circles around each predicted point
    for cx, cy in zip(px, py):
        fig.add_shape(
            type="circle",
            xref="x", yref="y",
            x0=cx - circle_mm, x1=cx + circle_mm,
            y0=cy - circle_mm, y1=cy + circle_mm,
            line=dict(width=1, dash="dot"),
            layer="above"
        )

    fig.update_layout(
        title=f"Next-step (x,y) prediction with {circle_mm} mm rings — sid={sid}, landmark={landmark}",
        xaxis=dict(title="x (mm)", range=[xmin, xmax]),
        yaxis=dict(title="y (mm)", range=[ymin, ymax], scaleanchor="x", scaleratio=1),  # keep circles round
        template="plotly_white",
        height=600,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0)
    )

    return fig


In [58]:
landmarks = ['pogonion', 'ans']
for l in landmarks:
    print(l)
    ans_female = female_df[female_df['landmark']==l]
    model_x, info_x, meta_x= fit_nextx_regressor(ans_female, use_dt=True, alpha=1.0, cv_splits=5)
    print(info_x) 
    model_y, info_y, meta_y= fit_nextx_regressor(ans_female, use_dt=True, alpha=1.0, cv_splits=5, x_col='y')
    print(info_y) 
    fig = plot_subject_landmark_nextxy(ans_female, model_x,model_y, sid="005", landmark=l, use_dt=True)
    fig.show()

pogonion
{'cv_MAE(mean)': 1.718370182620029, 'cv_MAE(std)': 0.15638008061304065, 'n_pairs': 874}
{'cv_MAE(mean)': 1.395651535972974, 'cv_MAE(std)': 0.07544367408125183, 'n_pairs': 874}


ans
{'cv_MAE(mean)': 1.3147086423334073, 'cv_MAE(std)': 0.16063773742838827, 'n_pairs': 874}
{'cv_MAE(mean)': 1.0176762337754355, 'cv_MAE(std)': 0.09221517557357277, 'n_pairs': 874}
